# 4C — Fast combined thermal + ESN optimization

This notebook estimates effusivity across 30–60 °C using **one combined predictor table**: manually computed thermal descriptors plus statistical summaries of ESN trajectories.

Requested constraints are enforced: **washout = 0** and **reservoir size ≤ 30**. To reduce runtime, candidate selection uses 5 material-grouped folds and one reservoir seed; the selected configuration is then evaluated once using the complete 14-fold LOMO protocol and three seeds.

The fast search is a screening/selection stage. The final LOMO section is the result to report.

## 1. Imports and fast-run configuration

For an even quicker exploratory run, reduce `SEARCH_N_TRIALS` to 15. Keep 30 or more for the result you intend to report.

In [1]:
from pathlib import Path
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import linalg
from scipy.signal import savgol_filter
from scipy.stats import kurtosis, skew
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

try:
    import optuna
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires Optuna. Install it in the active kernel with "
        "`%pip install optuna`, then restart the kernel."
    ) from exc

optuna.logging.set_verbosity(optuna.logging.WARNING)


/Users/markkeanujamesexconde/.virtualenvs/warmth-sensor-object-detection/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TEMPERATURE_FOLDERS = ("30C", "40C", "50C", "60C")
OPTIMIZATION_TARGET = "eff"
RANDOM_STATE = 42

ANALYSIS_WINDOW = (0.0, 5.0)
EARLY_WINDOW = (0.0, 1.0)
MID_WINDOW = (1.0, 3.0)
LATE_WINDOW = (3.0, 5.0)

# Requested constraints.
ESN_WASHOUT = 0
MIN_RES_SIZE = 8
MAX_RES_SIZE = 30

# Fast search: 30 candidates × 5 grouped folds × 1 seed = 150 model fits.
# The final evaluation uses 14 LOMO folds × 3 seeds = 42 additional fits.
SEARCH_N_TRIALS = 30
SEARCH_N_STARTUP_TRIALS = 8
SEARCH_N_SPLITS = 5
SEARCH_RESERVOIR_SEEDS = (42,)
FINAL_RESERVOIR_SEEDS = (42, 43, 44)
ENABLE_PRUNING = True
OPTUNA_SEED = 4042
STABILITY_WEIGHT = 0.25

# Defaults used outside a nominated Optuna candidate.
ESN_RES_SIZE = 20
ESN_LEAK_RATE = 0.1
ESN_INPUT_MAGNITUDE = 1.0
ESN_SPECTRAL_RADIUS = 0.9
XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 3, "learning_rate": 0.05,
    "subsample": 0.85, "colsample_bytree": 0.85,
    "min_child_weight": 2.0, "reg_alpha": 1e-4, "reg_lambda": 1.0,
}

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, cwd.parent) if (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
DATA_ROOT = PROJECT_ROOT / "data" / "02_preprocessed"
missing = [name for name in TEMPERATURE_FOLDERS if not (DATA_ROOT / name).is_dir()]
if missing:
    raise FileNotFoundError(f"Missing temperature data folders: {missing}")

RESULTS_DIR = PROJECT_ROOT / "results" / "multi_temp_esn"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
temperature_tag = "-".join(TEMPERATURE_FOLDERS)
RESULT_STEM = f"{temperature_tag}_{OPTIMIZATION_TARGET}_fast_combined_thermal_esn"
PARAMETER_FILE = RESULTS_DIR / f"{RESULT_STEM}_parameters.json"

print("Data root:", DATA_ROOT)
print("Search budget:", SEARCH_N_TRIALS, "trials ×", SEARCH_N_SPLITS, "folds × 1 seed")
print("Reservoir search range:", (MIN_RES_SIZE, MAX_RES_SIZE))
print("Washout fixed at:", ESN_WASHOUT)


Data root: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/data/02_preprocessed
Search budget: 30 trials × 5 folds × 1 seed
Reservoir search range: (8, 30)
Washout fixed at: 0


## 2. Load the four-temperature dataset and standard material properties

In [3]:
STANDARD_PROPERTIES = {
    "ps_foam":   {"k": 0.034, "Mass": 1.0, "Volume": 1.0, "rho": 25.0,   "cp": 1400.0},
    "pu_foam":   {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 30.0,   "cp": 1400.0},
    "cork":      {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 240.0,  "cp": 1800.0},
    "wood":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 700.0,  "cp": 1700.0},
    "pdms":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 970.0,  "cp": 1460.0},
    "gypsum":    {"k": 0.170, "Mass": 1.0, "Volume": 1.0, "rho": 800.0,  "cp": 1090.0},
    "cement":    {"k": 0.290, "Mass": 1.0, "Volume": 1.0, "rho": 1440.0, "cp": 750.0},
    "graphite":  {"k": 100.0, "Mass": 1.0, "Volume": 1.0, "rho": 1820.0, "cp": 710.0},
    "bismuth":   {"k": 8.1,   "Mass": 1.0, "Volume": 1.0, "rho": 9780.0, "cp": 130.0},
    "titanium":  {"k": 21.9,  "Mass": 1.0, "Volume": 1.0, "rho": 4506.0, "cp": 523.0},
    "nickel":    {"k": 90.9,  "Mass": 1.0, "Volume": 1.0, "rho": 8908.0, "cp": 461.0},
    "iron":      {"k": 80.4,  "Mass": 1.0, "Volume": 1.0, "rho": 7874.0, "cp": 449.0},
    "aluminum":  {"k": 237.0, "Mass": 1.0, "Volume": 1.0, "rho": 2700.0, "cp": 897.0},
    "copper":    {"k": 401.0, "Mass": 1.0, "Volume": 1.0, "rho": 8960.0, "cp": 385.0},
}

SAMPLE_ALIASES = {
    "ps": "ps_foam", "ps foam": "ps_foam", "ps_foam": "ps_foam",
    "pu": "pu_foam", "pu foam": "pu_foam", "pu_foam": "pu_foam",
    "cork": "cork", "cork fine": "cork", "cork_fine": "cork",
    "wood": "wood",
    "pdms": "pdms",
    "gypsum": "gypsum",
    "cement": "cement",
    "graphite": "graphite", "carbon": "graphite",
    "bi": "bismuth", "bismuth": "bismuth",
    "ti": "titanium", "titanium": "titanium",
    "ni": "nickel", "nickel": "nickel",
    "fe": "iron", "iron": "iron",
    "al": "aluminum", "aluminum": "aluminum",
    "cu": "copper", "copper": "copper",
}

standard_table = (
    pd.DataFrame.from_dict(STANDARD_PROPERTIES, orient="index")
    .rename_axis("Material")
    .reset_index()
)
display(standard_table)

required = {"Sample", "Trial", "Time", "Primary", "Secondary"}
frames = []
load_rows = []

for temperature in TEMPERATURE_FOLDERS:
    folder = DATA_ROOT / temperature
    for path in sorted(folder.glob("*.csv")):
        frame = pd.read_csv(path)
        missing = required - set(frame.columns)
        if missing:
            warnings.warn(
                f"Skipping {temperature}/{path.name}; missing {sorted(missing)}"
            )
            continue
        frame["Temperature"] = temperature
        frame["source_file"] = path.name
        frames.append(frame)
        load_rows.append({
            "Temperature": temperature,
            "file": path.name,
            "rows_loaded": len(frame),
        })

if not frames:
    raise ValueError("No valid trial files were found.")

DATA = pd.concat(frames, ignore_index=True)
DATA = DATA.replace([np.inf, -np.inf], np.nan)
for column in ("Trial", "Time", "Primary", "Secondary"):
    DATA[column] = pd.to_numeric(DATA[column], errors="coerce")
DATA = DATA.dropna(subset=list(required) + ["Temperature"]).copy()
DATA["Trial"] = DATA["Trial"].astype(int)
DATA["Temperature_C"] = pd.to_numeric(
    DATA["Temperature"].str.extract(r"(\d+(?:\.\d+)?)", expand=False),
    errors="coerce",
)
if DATA["Temperature_C"].isna().any():
    raise ValueError("A temperature-folder name could not be converted to Celsius.")

normalized_sample = (
    DATA["Sample"].astype(str).str.strip().str.lower().str.replace("_", " ")
)
DATA["Sample"] = normalized_sample.map(SAMPLE_ALIASES)
unknown_mask = DATA["Sample"].isna()
if unknown_mask.any():
    unknown = sorted(normalized_sample[unknown_mask].unique())
    raise KeyError(f"No standard-property mapping for samples: {unknown}")

# Ignore processed-file property cells and apply one standard table everywhere.
for property_name in ("k", "Mass", "Volume", "rho", "cp"):
    DATA[property_name] = DATA["Sample"].map(
        lambda sample: STANDARD_PROPERTIES[sample][property_name]
    )

# Temperature is required in the ID because material/trial numbers repeat by folder.
DATA["trial_id"] = (
    DATA["Temperature"].astype(str)
    + "__" + DATA["Sample"].astype(str)
    + "_trial_" + DATA["Trial"].astype(str)
)
DATA["eff"] = np.sqrt(DATA["k"] * DATA["rho"] * DATA["cp"])
DATA = DATA.sort_values(["Temperature", "trial_id", "Time"]).reset_index(drop=True)

summary = (
    DATA.groupby(["trial_id", "Temperature", "Sample", "Trial"], as_index=False)
    .agg(
        n_timesteps=("Time", "size"),
        k=("k", "first"),
        eff=("eff", "first"),
    )
)
temperature_summary = (
    summary.groupby("Temperature", as_index=False)
    .agg(
        trials=("trial_id", "nunique"),
        materials=("Sample", "nunique"),
        minimum_timesteps=("n_timesteps", "min"),
        maximum_timesteps=("n_timesteps", "max"),
    )
)
print(f"Rows: {len(DATA):,}")
print(f"Unique temperature-specific trials: {DATA['trial_id'].nunique()}")
display(temperature_summary)


,Material,k,Mass,Volume,rho,cp
0,ps_foam,0.034,1.0,1.0,25.0,1400.0
1,pu_foam,0.043,1.0,1.0,30.0,1400.0
2,cork,0.043,1.0,1.0,240.0,1800.0
3,wood,0.150,1.0,1.0,700.0,1700.0
4,pdms,0.150,1.0,1.0,970.0,1460.0
5,gypsum,0.170,1.0,1.0,800.0,1090.0
6,cement,0.290,1.0,1.0,1440.0,750.0
7,graphite,100.000,1.0,1.0,1820.0,710.0
8,bismuth,8.100,1.0,1.0,9780.0,130.0
9,titanium,21.900,1.0,1.0,4506.0,523.0


Rows: 70,480
Unique temperature-specific trials: 336


,Temperature,trials,materials,minimum_timesteps,maximum_timesteps
0,30C,84,14,184,227
1,40C,84,14,200,362
2,50C,84,14,183,404
3,60C,84,14,186,326


## 3. Detect contact and align every trial to the 0–5 s analysis window

In [4]:
def find_contact_time(
    trial,
    smooth_window=15,
    polyorder=2,
    threshold_frac=0.30,
    skip_samples=5,
):
    clean = (
        trial[["Time", "Primary"]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
        .sort_values("Time")
        .drop_duplicates("Time")
        .reset_index(drop=True)
    )
    time = clean["Time"].to_numpy(float)
    signal = clean["Primary"].to_numpy(float)
    if len(signal) < skip_samples + 7 or np.any(np.diff(time) <= 0):
        raise ValueError("Insufficient or invalid time samples.")

    work_time = time[skip_samples:]
    work_signal = signal[skip_samples:]
    window = min(int(smooth_window), len(work_signal))
    if window % 2 == 0:
        window -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    if window < minimum:
        raise ValueError("Sequence is too short for smoothing.")

    smooth = savgol_filter(work_signal, window, polyorder, mode="interp")
    derivative = np.gradient(smooth, work_time)
    strongest = int(np.argmin(derivative))
    active = derivative < threshold_frac * derivative[strongest]
    elbow = 0
    for position in range(strongest, -1, -1):
        if not active[position]:
            elbow = position + 1
            break
    return float(work_time[elbow])


In [5]:
aligned_trials = {}
alignment_rows = []

for trial_id, trial in DATA.groupby("trial_id", sort=False):
    trial = trial.sort_values("Time").drop_duplicates("Time").copy()
    try:
        contact_time = find_contact_time(trial)
    except ValueError as exc:
        warnings.warn(f"Skipping {trial_id}: {exc}")
        continue

    trial["time_from_contact"] = trial["Time"] - contact_time
    start, end = ANALYSIS_WINDOW
    trial = trial[
        trial["time_from_contact"].between(start, end, inclusive="both")
    ].copy()
    if len(trial) < 10:
        warnings.warn(f"Skipping {trial_id}: too few post-contact samples")
        continue
    aligned_trials[trial_id] = trial.reset_index(drop=True)
    alignment_rows.append({
        "trial_id": trial_id,
        "Temperature": trial["Temperature"].iloc[0],
        "Temperature_C": float(trial["Temperature_C"].iloc[0]),
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "contact_time": contact_time,
        "n_analysis_samples": len(trial),
    })

ALIGNMENT = pd.DataFrame(alignment_rows)
print(f"Aligned trials retained: {len(aligned_trials)}")
display(ALIGNMENT.head())


Aligned trials retained: 336


,trial_id,Temperature,Temperature_C,Sample,Trial,contact_time,n_analysis_samples
0,30C__aluminum_trial_1,30C,30.0,aluminum,1,8.40066,50
1,30C__aluminum_trial_2,30C,30.0,aluminum,2,7.40070,50
2,30C__aluminum_trial_3,30C,30.0,aluminum,3,7.70070,51
3,30C__aluminum_trial_4,30C,30.0,aluminum,4,8.70070,51
4,30C__aluminum_trial_5,30C,30.0,aluminum,5,7.20070,51


## 4. Compute the manual thermal feature family once

In [6]:
def _smooth(values, window=11, polyorder=2):
    values = np.asarray(values, float)
    selected = min(window, len(values))
    if selected % 2 == 0:
        selected -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    return (
        savgol_filter(values, selected, polyorder, mode="interp")
        if selected >= minimum else values.copy()
    )


def _slope(time, values, window):
    mask = (time >= window[0]) & (time <= window[1])
    if mask.sum() < 3:
        return np.nan
    return float(np.polyfit(time[mask], values[mask], 1)[0])


def extract_thermal_features(trial):
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))
    difference = primary - secondary
    result = {}

    for name, signal in {
        "primary": primary,
        "secondary": secondary,
        "difference": difference,
    }.items():
        change = signal - signal[0]
        rate = np.gradient(signal, time)
        result[f"{name}_final_change"] = float(change[-1])
        result[f"{name}_max_abs_change"] = float(np.max(np.abs(change)))
        result[f"{name}_response_auc"] = float(np.trapezoid(np.abs(change), time))
        result[f"{name}_early_slope"] = _slope(time, signal, EARLY_WINDOW)
        result[f"{name}_mid_slope"] = _slope(time, signal, MID_WINDOW)
        result[f"{name}_late_slope"] = _slope(time, signal, LATE_WINDOW)
        result[f"{name}_max_abs_rate"] = float(np.max(np.abs(rate)))
        result[f"{name}_rate_auc"] = float(np.trapezoid(np.abs(rate), time))

    # Dimensionless/cross-sensor summaries.
    primary_auc = result["primary_response_auc"]
    result["secondary_primary_auc_ratio"] = (
        result["secondary_response_auc"] / primary_auc
        if not np.isclose(primary_auc, 0) else np.nan
    )
    result["initial_sensor_difference"] = float(difference[0])
    result["final_sensor_difference"] = float(difference[-1])
    return result


thermal_rows = []
for trial_id, trial in aligned_trials.items():
    features = extract_thermal_features(trial)
    features.update({
        "trial_id": trial_id,
        "Temperature": trial["Temperature"].iloc[0],
        "Temperature_C": float(trial["Temperature_C"].iloc[0]),
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "k": float(trial["k"].iloc[0]),
        "eff": float(trial["eff"].iloc[0]),
    })
    thermal_rows.append(features)

THERMAL_FEATURES = pd.DataFrame(thermal_rows)
print(f"Thermal features: {len(THERMAL_FEATURES.columns) - 7}")
display(THERMAL_FEATURES.head())


Thermal features: 27


,primary_final_change,primary_max_abs_change,primary_response_auc,primary_early_slope,primary_mid_slope,primary_late_slope,primary_max_abs_rate,primary_rate_auc,secondary_final_change,secondary_max_abs_change,...,secondary_primary_auc_ratio,initial_sensor_difference,final_sensor_difference,trial_id,Temperature,Temperature_C,Sample,Trial,k,eff
0,-0.000003,0.000003,0.000013,-0.000002,-3.654261e-07,-1.344399e-07,0.000003,0.000003,-0.000002,0.000002,...,0.225534,-0.000040,-0.000042,30C__aluminum_trial_1,30C,30.0,aluminum,1,237.0,23958.094665
1,-0.000003,0.000003,0.000014,-0.000002,-3.595754e-07,-1.291998e-07,0.000004,0.000003,-0.000002,0.000002,...,0.241267,-0.000040,-0.000042,30C__aluminum_trial_2,30C,30.0,aluminum,2,237.0,23958.094665
2,-0.000003,0.000003,0.000014,-0.000002,-3.389437e-07,-1.112019e-07,0.000003,0.000003,-0.000002,0.000002,...,0.253393,-0.000040,-0.000042,30C__aluminum_trial_3,30C,30.0,aluminum,3,237.0,23958.094665
3,-0.000003,0.000003,0.000013,-0.000002,-3.724766e-07,-1.181206e-07,0.000003,0.000003,-0.000002,0.000002,...,0.243614,-0.000040,-0.000042,30C__aluminum_trial_4,30C,30.0,aluminum,4,237.0,23958.094665
4,-0.000003,0.000003,0.000013,-0.000002,-3.853054e-07,-1.244971e-07,0.000003,0.000003,-0.000002,0.000002,...,0.264204,-0.000041,-0.000042,30C__aluminum_trial_5,30C,30.0,aluminum,5,237.0,23958.094665


## 5. Define the five ESN inputs, reservoir, and trajectory summaries

In [7]:
class ManualReservoir:
    def __init__(
        self,
        res_size=30,
        leak_rate=0.9,
        input_magnitude=1.5,
        spectral_radius=1.3,
        washout=0,
        random_state=42,
    ):
        self.res_size = int(res_size)
        self.leak_rate = float(leak_rate)
        self.input_magnitude = float(input_magnitude)
        self.spectral_radius = float(spectral_radius)
        self.washout = int(washout)
        rng = np.random.default_rng(random_state)
        self.Win = (rng.random((self.res_size, 1 + 5)) - 0.5) * self.input_magnitude
        W = rng.random((self.res_size, self.res_size)) - 0.5
        radius = np.max(np.abs(linalg.eigvals(W)))
        if not np.isfinite(radius) or np.isclose(radius, 0):
            raise ValueError("Invalid reservoir spectral radius.")
        self.W = (W / radius.real) * self.spectral_radius

    def run(self, sequence):
        sequence = np.asarray(sequence, float)
        if sequence.ndim != 2 or sequence.shape[1] != 5:
            raise ValueError("Expected sequence with five input channels.")
        if len(sequence) <= self.washout:
            raise ValueError(
                f"Sequence has {len(sequence)} samples, but washout={self.washout}. "
                "Washout must be smaller than the sequence length."
            )
        x = np.zeros((self.res_size, 1))
        states = []
        for row in sequence:
            u = row.reshape(-1, 1)
            x = (
                (1 - self.leak_rate) * x
                + self.leak_rate * np.tanh(
                    self.Win @ np.vstack((1.0, u)) + self.W @ x
                )
            )
            states.append(x[:, 0].copy())
        return np.asarray(states)[self.washout:]


def raw_esn_input(trial):
    """Five post-contact channels after per-trial baseline centering.

    Centering uses only the trial's first retained sensor sample and never uses
    k, eff, another material, or a future target value. The fold-local
    StandardScaler in build_esn_feature_table remains the model scale transform.
    """
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))

    primary = primary - primary[0]
    secondary = secondary - secondary[0]
    difference = primary - secondary
    primary_rate = np.gradient(primary, time)
    secondary_rate = np.gradient(secondary, time)
    return np.column_stack([
        primary, secondary, difference, primary_rate, secondary_rate
    ])


def summarize_reservoir_trajectories(time, states):
    """Return finite statistical/dynamical summaries for every reservoir unit."""
    time = np.asarray(time, float)
    states = np.asarray(states, float)
    if states.ndim != 2 or len(time) != len(states):
        raise ValueError("Time and reservoir states must have matching rows.")
    if len(time) < 2 or np.any(~np.isfinite(time)) or np.any(np.diff(time) <= 0):
        raise ValueError("Reservoir-state time must be finite and strictly increasing.")
    if np.any(~np.isfinite(states)):
        raise ValueError("Reservoir states contain non-finite values.")

    result = {}
    for unit in range(states.shape[1]):
        values = states[:, unit]
        maximum_absolute_index = int(np.argmax(np.abs(values)))
        standard_deviation = float(np.std(values, ddof=0))

        # Constant or nearly constant trajectories have well-defined zero shape
        # rather than scipy's otherwise undefined skewness/kurtosis warnings.
        if len(values) < 3 or np.isclose(standard_deviation, 0.0):
            skewness = 0.0
        else:
            skewness = float(skew(values, bias=False))
        if len(values) < 4 or np.isclose(standard_deviation, 0.0):
            excess_kurtosis = 0.0
        else:
            excess_kurtosis = float(kurtosis(values, fisher=True, bias=False))

        summaries = {
            "mean": float(np.mean(values)),
            "std": standard_deviation,
            "range": float(np.ptp(values)),
            "net_change": float(values[-1] - values[0]),
            "slope": float(np.polyfit(time, values, 1)[0]),
            "absolute_area": float(np.trapezoid(np.abs(values), time)),
            "time_of_max_absolute": float(time[maximum_absolute_index]),
            "skewness": skewness,
            "excess_kurtosis": excess_kurtosis,
        }
        
        for name, value in summaries.items():
            if not np.isfinite(value):
                raise ValueError(f"Non-finite {name} for reservoir unit {unit}.")
            result[f"esn_{name}_u{unit:03d}"] = value
    return result


## 6. Regression metrics and XGBoost model

In [8]:
def regression_metrics(y_true, y_pred):
    """Metrics requiring variation in y_true; intended for pooled or mixed-target data."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) == 0 or np.any(~np.isfinite(y_true)) or np.any(~np.isfinite(y_pred)):
        raise ValueError("Metrics require non-empty, finite targets and predictions.")
    target_range = float(np.ptp(y_true))
    if len(y_true) < 2 or target_range <= 0:
        raise ValueError(
            "R² and test-range NRMSE require at least two distinct target values. "
            "Use constant_target_fold_metrics for a single-material LOMO fold."
        )
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "n": len(y_true),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "nrmse_range": rmse / target_range,
        "r2": float(r2_score(y_true, y_pred)),
        "median_ape_pct": float(np.median(np.abs((y_true - y_pred) / y_true)) * 100),
    }


def constant_target_fold_metrics(y_true, y_pred, training_target_range):
    """Valid diagnostics for one outer LOMO fold with a constant test target."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    training_target_range = float(training_target_range)
    if len(y_true) == 0 or np.any(~np.isfinite(y_true)) or np.any(~np.isfinite(y_pred)):
        raise ValueError("Fold metrics require non-empty, finite values.")
    if training_target_range <= 0:
        raise ValueError("The outer-training target range must be positive.")
    residual = y_pred - y_true
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return {
        "n": len(y_true),
        "mae": float(np.mean(np.abs(residual))),
        "rmse": rmse,
        "nrmse_training_range": rmse / training_target_range,
        "mean_error_bias": float(np.mean(residual)),
        "median_ape_pct": float(np.median(np.abs(residual / y_true)) * 100),
    }


def make_regressor(xgb_params=None, random_state=RANDOM_STATE):
    params = {**XGB_PARAMS, **(xgb_params or {})}
    xgb = XGBRegressor(
        objective="reg:squarederror",
        random_state=int(random_state),
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
        **params,
    )
    base = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("xgb", xgb),
    ])
    return TransformedTargetRegressor(
        regressor=base,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


## 7. Construct the combined thermal + summarized-ESN predictor table

In [9]:
def build_esn_feature_table(train_ids, all_ids, esn_params=None, random_state=RANDOM_STATE):
    params = {
        "res_size": ESN_RES_SIZE,
        "leak_rate": ESN_LEAK_RATE,
        "input_magnitude": ESN_INPUT_MAGNITUDE,
        "spectral_radius": ESN_SPECTRAL_RADIUS,
        "washout": ESN_WASHOUT,
    }
    params.update(esn_params or {})
    scaler = StandardScaler().fit(np.vstack([
        raw_esn_input(aligned_trials[trial_id]) for trial_id in train_ids
    ]))
    reservoir = ManualReservoir(**params, random_state=random_state)
    rows = []
    for trial_id in all_ids:
        trial = aligned_trials[trial_id]
        sequence = scaler.transform(raw_esn_input(trial))
        states = reservoir.run(sequence)
        state_time = trial["time_from_contact"].to_numpy(float)[params["washout"]:]
        features = summarize_reservoir_trajectories(state_time, states)
        features.update({
            "trial_id": trial_id,
            "Temperature": trial["Temperature"].iloc[0],
            "Temperature_C": float(trial["Temperature_C"].iloc[0]),
            "Sample": trial["Sample"].iloc[0],
            "Trial": int(trial["Trial"].iloc[0]),
            "k": float(trial["k"].iloc[0]),
            "eff": float(trial["eff"].iloc[0]),
        })
        rows.append(features)
    return pd.DataFrame(rows)

In [10]:
METADATA_COLUMNS = [
    "trial_id", "Temperature", "Temperature_C", "Sample", "Trial", "k", "eff"
]
THERMAL_PREDICTORS = [
    column for column in THERMAL_FEATURES.columns
    if column not in METADATA_COLUMNS
]


def build_combined_feature_table(train_ids, all_ids, esn_params, random_state):
    """Create fold-safe ESN summaries and append precomputed thermal descriptors."""
    esn_table = build_esn_feature_table(
        train_ids, all_ids, esn_params=esn_params, random_state=random_state
    )
    thermal_part = THERMAL_FEATURES[["trial_id", *THERMAL_PREDICTORS]].copy()
    combined = esn_table.merge(thermal_part, on="trial_id", how="left", validate="one_to_one")
    if combined[THERMAL_PREDICTORS].isna().all(axis=None):
        raise ValueError("All manually computed thermal features are missing after merge.")
    return combined


def predictor_columns(frame):
    # Temperature is an operating-condition predictor. Targets and IDs are excluded.
    esn_columns = [c for c in frame.columns if c.startswith("esn_")]
    return ["Temperature_C", *THERMAL_PREDICTORS, *esn_columns]


def split_parameters(params):
    esn_keys = {"res_size", "leak_rate", "input_magnitude", "spectral_radius", "washout"}
    return (
        {key: params[key] for key in esn_keys},
        {key: value for key, value in params.items() if key not in esn_keys},
    )


def predict_fold(train_meta, valid_meta, params, seeds):
    esn_params, xgb_params = split_parameters(params)
    train_ids = train_meta["trial_id"].tolist()
    valid_ids = valid_meta["trial_id"].tolist()
    all_ids = train_ids + valid_ids
    seed_predictions = []

    for seed in seeds:
        features = build_combined_feature_table(train_ids, all_ids, esn_params, seed)
        features = features.set_index("trial_id")
        columns = predictor_columns(features.reset_index())
        X_train = features.loc[train_ids, columns]
        X_valid = features.loc[valid_ids, columns]
        y_train = train_meta.set_index("trial_id").loc[train_ids, OPTIMIZATION_TARGET]
        model = make_regressor(xgb_params=xgb_params, random_state=seed)
        model.fit(X_train, y_train)
        seed_predictions.append(model.predict(X_valid))

    return np.mean(np.vstack(seed_predictions), axis=0)


## 8. Fast grouped Optuna search

Each candidate is tested on the same five material-grouped folds. No material appears in both training and validation within a fold. Only seed 42 is used here to make selection inexpensive. Median pruning may stop weak candidates after three folds.

In [11]:
def suggest_parameters(trial):
    return {
        "res_size": trial.suggest_int("res_size", MIN_RES_SIZE, MAX_RES_SIZE),
        "leak_rate": trial.suggest_float("leak_rate", 0.05, 0.95),
        "input_magnitude": trial.suggest_float("input_magnitude", 0.25, 2.0),
        "spectral_radius": trial.suggest_float("spectral_radius", 0.30, 1.50),
        "washout": 0,
        # Narrower XGBoost space keeps poor, very large candidates from wasting time.
        "n_estimators": trial.suggest_int("n_estimators", 150, 600),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.2, 8.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.70, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.03, 20.0, log=True),
    }


METADATA = THERMAL_FEATURES[METADATA_COLUMNS].reset_index(drop=True)
group_cv = GroupKFold(n_splits=SEARCH_N_SPLITS)
search_splits = list(group_cv.split(METADATA, groups=METADATA["Sample"]))


def objective(trial):
    params = suggest_parameters(trial)
    started = time.perf_counter()
    prediction_parts = []
    fold_nrmse = []

    for fold, (train_idx, valid_idx) in enumerate(search_splits, start=1):
        train_meta = METADATA.iloc[train_idx].reset_index(drop=True)
        valid_meta = METADATA.iloc[valid_idx].reset_index(drop=True)
        y_true = valid_meta[OPTIMIZATION_TARGET].to_numpy(float)
        y_pred = predict_fold(train_meta, valid_meta, params, SEARCH_RESERVOIR_SEEDS)
        metrics = regression_metrics(y_true, y_pred)
        fold_nrmse.append(metrics["nrmse_range"])
        prediction_parts.append(pd.DataFrame({"y_true": y_true, "y_pred": y_pred}))

        partial = pd.concat(prediction_parts, ignore_index=True)
        partial_range = float(np.ptp(partial["y_true"]))
        partial_rmse = float(np.sqrt(mean_squared_error(partial["y_true"], partial["y_pred"])))
        partial_score = partial_rmse / partial_range + STABILITY_WEIGHT * np.std(fold_nrmse)
        trial.report(float(partial_score), step=fold)
        if ENABLE_PRUNING and fold >= 3 and trial.should_prune():
            raise optuna.TrialPruned()

    pooled_predictions = pd.concat(prediction_parts, ignore_index=True)
    pooled = regression_metrics(pooled_predictions["y_true"], pooled_predictions["y_pred"])
    objective_j = float(pooled["nrmse_range"] + STABILITY_WEIGHT * np.std(fold_nrmse))
    trial.set_user_attr("pooled_nrmse", pooled["nrmse_range"])
    trial.set_user_attr("pooled_r2", pooled["r2"])
    trial.set_user_attr("elapsed_minutes", (time.perf_counter() - started) / 60)
    print(
        f"Trial {trial.number + 1:02d}/{SEARCH_N_TRIALS}: J={objective_j:.4f}, "
        f"NRMSE={pooled['nrmse_range']:.4f}, R²={pooled['r2']:.4f}, "
        f"elapsed={(time.perf_counter() - started) / 60:.1f} min"
    )
    return objective_j


sampler = optuna.samplers.TPESampler(
    n_startup_trials=SEARCH_N_STARTUP_TRIALS,
    multivariate=True,
    seed=OPTUNA_SEED,
)
pruner = (
    optuna.pruners.MedianPruner(
        n_startup_trials=SEARCH_N_STARTUP_TRIALS, n_warmup_steps=3
    )
    if ENABLE_PRUNING else optuna.pruners.NopPruner()
)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.enqueue_trial({
    "res_size": 20, "leak_rate": 0.1, "input_magnitude": 1.0,
    "spectral_radius": 0.9, "n_estimators": 300, "max_depth": 3,
    "learning_rate": 0.05, "min_child_weight": 2.0,
    "subsample": 0.85, "colsample_bytree": 0.85,
    "reg_alpha": 1e-4, "reg_lambda": 1.0,
})
study.optimize(objective, n_trials=SEARCH_N_TRIALS, gc_after_trial=True)

BEST_PARAMETERS = dict(study.best_params)
BEST_PARAMETERS["washout"] = 0
BEST_PARAMETERS["res_size"] = int(BEST_PARAMETERS["res_size"])
BEST_PARAMETERS["n_estimators"] = int(BEST_PARAMETERS["n_estimators"])
BEST_PARAMETERS["max_depth"] = int(BEST_PARAMETERS["max_depth"])
SEARCH_HISTORY = study.trials_dataframe()

print("Best fast-search objective:", study.best_value)
display(pd.Series(BEST_PARAMETERS, name="selected value"))
display(SEARCH_HISTORY.sort_values("value").head(10))


/var/folders/fx/p7h99klx61g3f9j63ld4_xcm0000gn/T/ipykernel_51700/1321533735.py:62: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(


Trial 01/30: J=0.3843, NRMSE=0.2356, R²=0.3901, elapsed=0.3 min
Trial 02/30: J=0.3724, NRMSE=0.2323, R²=0.4070, elapsed=0.3 min
Trial 03/30: J=0.4086, NRMSE=0.2461, R²=0.3341, elapsed=0.4 min
Trial 04/30: J=0.3754, NRMSE=0.2398, R²=0.3680, elapsed=0.4 min
Trial 05/30: J=0.3826, NRMSE=0.2418, R²=0.3573, elapsed=0.1 min
Trial 06/30: J=0.3846, NRMSE=0.2357, R²=0.3896, elapsed=0.3 min
Trial 07/30: J=0.4350, NRMSE=0.2517, R²=0.3036, elapsed=0.2 min
Trial 08/30: J=0.3577, NRMSE=0.2259, R²=0.4393, elapsed=0.2 min
Trial 09/30: J=0.3842, NRMSE=0.2380, R²=0.3777, elapsed=0.2 min
Trial 10/30: J=0.3621, NRMSE=0.2383, R²=0.3761, elapsed=0.2 min
Trial 11/30: J=0.3720, NRMSE=0.2296, R²=0.4204, elapsed=0.2 min
Trial 12/30: J=0.3437, NRMSE=0.2359, R²=0.3884, elapsed=0.2 min
Trial 13/30: J=0.3454, NRMSE=0.2260, R²=0.4388, elapsed=0.2 min
Trial 14/30: J=0.3759, NRMSE=0.2331, R²=0.4028, elapsed=0.2 min
Trial 15/30: J=0.3937, NRMSE=0.2363, R²=0.3862, elapsed=0.1 min
Trial 16/30: J=0.3716, NRMSE=0.2319, R²=

res_size             16.000000
leak_rate             0.852332
input_magnitude       1.920875
spectral_radius       0.575442
n_estimators        381.000000
max_depth             2.000000
learning_rate         0.076671
min_child_weight      1.384272
subsample             0.749249
colsample_bytree      0.602401
reg_alpha             0.000006
reg_lambda            0.184577
washout               0.000000
Name: selected value, dtype: float64

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_input_magnitude,params_leak_rate,params_learning_rate,params_max_depth,...,params_reg_alpha,params_reg_lambda,params_res_size,params_spectral_radius,params_subsample,user_attrs_elapsed_minutes,user_attrs_pooled_nrmse,user_attrs_pooled_r2,system_attrs_fixed_params,state
11,11,0.343686,2026-08-25 21:51:40.352912,2026-08-25 21:51:51.835758,0 days 00:00:11.482846,0.602401,1.920875,0.852332,0.076671,2,...,0.000006,0.184577,16,0.575442,0.749249,0.191337,0.235895,0.388390,NaN,COMPLETE
12,12,0.345396,2026-08-25 21:51:51.880472,2026-08-25 21:52:04.337907,0 days 00:00:12.457435,0.748210,1.915121,0.680084,0.048478,2,...,0.000928,0.030104,18,0.546985,0.751329,0.207579,0.225967,0.438792,NaN,COMPLETE
25,25,0.349609,2026-08-25 21:54:34.736901,2026-08-25 21:54:49.027960,0 days 00:00:14.291059,0.602839,1.993098,0.815823,0.061283,2,...,0.000030,1.541502,18,0.626196,0.746739,0.237959,0.228186,0.427713,NaN,COMPLETE
18,18,0.350413,2026-08-25 21:53:03.579911,2026-08-25 21:53:15.490552,0 days 00:00:11.910641,0.608299,1.066666,0.617054,0.043542,2,...,0.000016,0.816727,16,0.667556,0.768645,0.198459,0.230439,0.416355,NaN,COMPLETE
27,27,0.352342,2026-08-25 21:55:00.666110,2026-08-25 21:55:16.516234,0 days 00:00:15.850124,0.618530,1.805145,0.681852,0.050355,2,...,0.000063,2.951540,24,0.568255,0.783224,0.264118,0.226830,0.434493,NaN,COMPLETE
28,28,0.353662,2026-08-25 21:55:16.556268,2026-08-25 21:55:26.712221,0 days 00:00:10.155953,0.614022,1.732014,0.584436,0.069986,2,...,0.000013,1.545753,14,0.352275,0.710476,0.169216,0.230096,0.418092,NaN,COMPLETE
7,7,0.357692,2026-08-25 21:50:54.684162,2026-08-25 21:51:06.859038,0 days 00:00:12.174876,0.656188,1.275920,0.894406,0.052451,4,...,0.000001,0.032746,14,0.428293,0.780958,0.202893,0.225859,0.439325,NaN,COMPLETE
17,17,0.358683,2026-08-25 21:52:54.191926,2026-08-25 21:53:03.482918,0 days 00:00:09.290992,0.873842,1.397785,0.829745,0.086666,2,...,0.000002,3.006112,12,0.438043,0.812752,0.154802,0.232826,0.404205,NaN,COMPLETE
21,21,0.360350,2026-08-25 21:53:46.004112,2026-08-25 21:53:56.721817,0 days 00:00:10.717705,0.604356,0.886852,0.582976,0.055308,2,...,0.000034,0.205276,13,0.651193,0.850622,0.178576,0.233949,0.398440,NaN,COMPLETE
22,22,0.360630,2026-08-25 21:53:56.772039,2026-08-25 21:54:08.282148,0 days 00:00:11.510109,0.657053,0.909044,0.750616,0.035707,3,...,0.000228,3.828056,13,0.406875,0.776920,0.191779,0.228208,0.427605,NaN,COMPLETE


## 9. Final full LOMO evaluation

This section freezes the selected hyperparameters, holds out each material once, builds fold-local ESN features with seeds 42/43/44, averages the three predictions per trial, and pools all 336 out-of-fold predictions. These are the performance values to report.

In [12]:
logo = LeaveOneGroupOut()
fold_rows = []
prediction_rows = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(METADATA, groups=METADATA["Sample"]), start=1
):
    train_meta = METADATA.iloc[train_idx].reset_index(drop=True)
    test_meta = METADATA.iloc[test_idx].reset_index(drop=True)
    held_out = str(test_meta["Sample"].iloc[0])
    y_true = test_meta[OPTIMIZATION_TARGET].to_numpy(float)
    y_pred = predict_fold(train_meta, test_meta, BEST_PARAMETERS, FINAL_RESERVOIR_SEEDS)
    diagnostics = constant_target_fold_metrics(
        y_true, y_pred, np.ptp(train_meta[OPTIMIZATION_TARGET].to_numpy(float))
    )
    fold_rows.append({"fold": fold, "held_out_material": held_out, **diagnostics})
    prediction_rows.append(pd.DataFrame({
        "trial_id": test_meta["trial_id"],
        "Temperature": test_meta["Temperature"],
        "Temperature_C": test_meta["Temperature_C"],
        "Sample": test_meta["Sample"],
        "Trial": test_meta["Trial"],
        "fold": fold,
        "y_true": y_true,
        "y_pred": y_pred,
    }))
    print(f"Final LOMO fold {fold:02d}/14: held out {held_out}")

FINAL_FOLD_METRICS = pd.DataFrame(fold_rows)
FINAL_OOF_PREDICTIONS = pd.concat(prediction_rows, ignore_index=True)
FINAL_POOLED_METRICS = pd.DataFrame([
    regression_metrics(FINAL_OOF_PREDICTIONS["y_true"], FINAL_OOF_PREDICTIONS["y_pred"])
])

print("Final fixed-parameter, three-seed, pooled LOMO performance:")
display(FINAL_POOLED_METRICS)
print("Per-material diagnostics (R² is intentionally not computed within constant-target folds):")
display(FINAL_FOLD_METRICS)


Final LOMO fold 01/14: held out aluminum
Final LOMO fold 02/14: held out bismuth
Final LOMO fold 03/14: held out cement
Final LOMO fold 04/14: held out copper
Final LOMO fold 05/14: held out cork
Final LOMO fold 06/14: held out graphite
Final LOMO fold 07/14: held out gypsum
Final LOMO fold 08/14: held out iron
Final LOMO fold 09/14: held out nickel
Final LOMO fold 10/14: held out pdms
Final LOMO fold 11/14: held out ps_foam
Final LOMO fold 12/14: held out pu_foam
Final LOMO fold 13/14: held out titanium
Final LOMO fold 14/14: held out wood
Final fixed-parameter, three-seed, pooled LOMO performance:


,n,mae,rmse,nrmse_range,r2,median_ape_pct
0,336,4532.300551,8191.977505,0.220463,0.465798,54.360179


Per-material diagnostics (R² is intentionally not computed within constant-target folds):


,fold,held_out_material,n,mae,rmse,nrmse_training_range,mean_error_bias,median_ape_pct
0,1,aluminum,24,8266.154505,9406.466629,0.253147,-8010.363403,37.087409
1,2,bismuth,24,8473.549340,9193.845363,0.247425,8473.549340,286.752221
2,3,cement,24,960.293841,1504.352225,0.040485,856.188734,42.137384
3,4,copper,24,24589.332555,24963.784602,1.043480,-24589.332555,66.097580
4,5,cork,24,120.867059,152.530524,0.004105,80.634968,76.176427
5,6,graphite,24,3074.898536,3902.786731,0.105032,396.106336,26.949961
6,7,gypsum,24,981.184804,1658.989929,0.044647,916.300549,46.155710
7,8,iron,24,4258.605598,5382.976781,0.144867,307.822244,19.863988
8,9,nickel,24,5872.765044,7053.775703,0.189831,-5828.087547,28.184397
9,10,pdms,24,229.544134,247.699732,0.006666,-229.544134,48.892768


## 10. Save the selected configuration and evaluation artifacts

In [13]:
SEARCH_HISTORY_FILE = RESULTS_DIR / f"{RESULT_STEM}_search_history.csv"
FOLD_METRICS_FILE = RESULTS_DIR / f"{RESULT_STEM}_fold_metrics.csv"
OOF_FILE = RESULTS_DIR / f"{RESULT_STEM}_oof_predictions.csv"
POOLED_FILE = RESULTS_DIR / f"{RESULT_STEM}_pooled_metrics.csv"

SEARCH_HISTORY.to_csv(SEARCH_HISTORY_FILE, index=False)
FINAL_FOLD_METRICS.to_csv(FOLD_METRICS_FILE, index=False)
FINAL_OOF_PREDICTIONS.to_csv(OOF_FILE, index=False)
FINAL_POOLED_METRICS.to_csv(POOLED_FILE, index=False)

esn_params, xgb_params = split_parameters(BEST_PARAMETERS)
payload = {
    "source": "4C fast combined manually computed thermal plus summarized ESN features",
    "temperature_folders": list(TEMPERATURE_FOLDERS),
    "target": OPTIMIZATION_TARGET,
    "analysis_window": list(ANALYSIS_WINDOW),
    "feature_family": "combined_thermal_and_summarized_esn",
    "thermal_predictors": THERMAL_PREDICTORS,
    "esn_summary_features": [
        "mean", "std", "range", "net_change", "slope", "absolute_area",
        "time_of_max_absolute", "skewness", "excess_kurtosis",
    ],
    "washout_fixed": 0,
    "reservoir_search_range": [MIN_RES_SIZE, MAX_RES_SIZE],
    "search_protocol": {
        "method": "Optuna multivariate TPE",
        "grouping": f"{SEARCH_N_SPLITS}-fold GroupKFold by material",
        "n_trials": SEARCH_N_TRIALS,
        "reservoir_seeds": list(SEARCH_RESERVOIR_SEEDS),
        "pruning_enabled": ENABLE_PRUNING,
    },
    "final_evaluation": {
        "protocol": "14-fold leave-one-material-out",
        "reservoir_seeds": list(FINAL_RESERVOIR_SEEDS),
        "prediction_aggregation": "mean across seed-specific regressors",
    },
    "esn_parameters": esn_params,
    "xgb_parameters": xgb_params,
    "final_pooled_metrics": FINAL_POOLED_METRICS.iloc[0].to_dict(),
}
with PARAMETER_FILE.open("w") as file:
    json.dump(payload, file, indent=2)

print("Saved parameter handoff:", PARAMETER_FILE)
print("Saved OOF predictions:", OOF_FILE)


Saved parameter handoff: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/30C-40C-50C-60C_eff_fast_combined_thermal_esn_parameters.json
Saved OOF predictions: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/30C-40C-50C-60C_eff_fast_combined_thermal_esn_oof_predictions.csv


## Runtime choices and their trade-offs

- **Five grouped search folds instead of 14 LOMO search folds:** faster candidate ranking while still preventing material leakage.
- **One search seed instead of three:** removes a 3× multiplier during optimization; three seeds return for final evaluation.
- **30 trials instead of 100:** large runtime reduction, with less exhaustive exploration.
- **Reservoir capped at 30 units:** fewer ESN states and fewer combined predictors.
- **Narrower XGBoost range and pruning:** avoids spending full time on very large or clearly weak candidates.
- **Manual thermal features computed once:** they are reused rather than regenerated for every candidate.

Do not interpret the five-fold search score as the final model performance. Report the pooled metrics from Section 9.